# 01 - EDA dos dados reais

Este notebook documenta a primeira leitura da base **Store Sales - Time Series Forecasting**, usada no dashboard de previsão de vendas e planejamento de receita.

O objetivo aqui não é treinar modelo ainda. A meta é entender a estrutura dos dados, a granularidade, os períodos disponíveis, o comportamento das vendas e os principais sinais que serão úteis para modelagem.

Ao final deste notebook, devemos conseguir responder:

- Qual é a granularidade da base?
- Quais arquivos são obrigatórios para o dashboard?
- Qual período histórico está disponível?
- Como vendas e promoções se comportam no tempo?
- Quais famílias e lojas concentram maior volume?
- Que cuidados precisam ser considerados antes de modelar?

## 1. Configuração do ambiente

As células abaixo localizam a raiz do projeto e adicionam `src/` ao `sys.path`, permitindo reutilizar os módulos do dashboard dentro do notebook.

Se você estiver executando pelo VS Code, selecione o interpretador do ambiente Poetry do projeto.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATA_DIR

## 2. Conferência dos arquivos

O dashboard usa obrigatoriamente:

- `train.csv`: histórico diário de vendas por loja e família.
- `stores.csv`: metadados das lojas.

Outros arquivos da competição também são úteis para análises e evolução futura do modelo, como `test.csv`, `transactions.csv`, `oil.csv` e `holidays_events.csv`.

In [ ]:
expected_files = [
    "train.csv",
    "test.csv",
    "stores.csv",
    "transactions.csv",
    "oil.csv",
    "holidays_events.csv",
]

file_status = pd.DataFrame(
    {
        "arquivo": expected_files,
        "existe": [(DATA_DIR / name).exists() for name in expected_files],
        "tamanho_mb": [
            round((DATA_DIR / name).stat().st_size / 1024 / 1024, 2)
            if (DATA_DIR / name).exists()
            else None
            for name in expected_files
        ],
    }
)
file_status

In [ ]:
missing_required = [name for name in ["train.csv", "stores.csv"] if not (DATA_DIR / name).exists()]
if missing_required:
    raise FileNotFoundError(
        f"Arquivos obrigatórios ausentes em {DATA_DIR}: {missing_required}. "
        "Execute scripts/download_store_sales.py antes de continuar."
    )

## 3. Leitura eficiente dos dados

A tabela `train.csv` é a maior do projeto. Por isso, vamos ler apenas as colunas necessárias para a análise inicial e definir tipos de dados mais econômicos.

A métrica `sales` será tratada como volume de vendas planejadas. A base não contém preço unitário ou margem, então não é correto chamar esse valor de receita financeira em moeda sem uma tabela adicional de preços ou ticket médio.

In [ ]:
train = pd.read_csv(
    DATA_DIR / "train.csv",
    usecols=["date", "store_nbr", "family", "sales", "onpromotion"],
    parse_dates=["date"],
    dtype={
        "store_nbr": "int16",
        "family": "category",
        "sales": "float32",
        "onpromotion": "int16",
    },
)

stores = pd.read_csv(DATA_DIR / "stores.csv")

train.head()

In [ ]:
overview = pd.DataFrame(
    {
        "tabela": ["train", "stores"],
        "linhas": [len(train), len(stores)],
        "colunas": [train.shape[1], stores.shape[1]],
        "memoria_mb": [
            round(train.memory_usage(deep=True).sum() / 1024 / 1024, 2),
            round(stores.memory_usage(deep=True).sum() / 1024 / 1024, 2),
        ],
    }
)
overview

## 4. Granularidade e período histórico

A granularidade principal é:

**data + loja + família de produto**

Essa é exatamente a granularidade usada pelo modelo para gerar previsões por frente comercial.

In [ ]:
period_summary = pd.Series(
    {
        "data_inicial": train["date"].min(),
        "data_final": train["date"].max(),
        "dias_no_historico": train["date"].nunique(),
        "lojas": train["store_nbr"].nunique(),
        "familias": train["family"].nunique(),
        "series_loja_familia": train[["store_nbr", "family"]].drop_duplicates().shape[0],
    }
)
period_summary

In [ ]:
duplicate_rows = train.duplicated(["date", "store_nbr", "family"]).sum()
missing_values = train.isna().sum().to_frame("valores_ausentes")

print(f"Registros duplicados na chave data + loja + família: {duplicate_rows:,}")
missing_values

## 5. Evolução temporal das vendas

Vamos agregar as vendas por dia para observar tendência, sazonalidade e possíveis períodos atípicos.

In [ ]:
daily_sales = (
    train.groupby("date", as_index=False)
    .agg(
        sales=("sales", "sum"),
        promoted_items=("onpromotion", "sum"),
    )
    .sort_values("date")
)

daily_sales["sales_7d_ma"] = daily_sales["sales"].rolling(7, min_periods=1).mean()
daily_sales["sales_28d_ma"] = daily_sales["sales"].rolling(28, min_periods=1).mean()

daily_sales.tail()

In [ ]:
fig = px.line(
    daily_sales,
    x="date",
    y=["sales", "sales_7d_ma", "sales_28d_ma"],
    title="Vendas diárias: valor real e médias móveis",
    labels={"value": "vendas", "date": "data", "variable": "série"},
)
fig.update_layout(hovermode="x unified")
fig.show()

## 6. Sazonalidade por dia da semana

Um dos sinais usados nos modelos é o comportamento médio por dia da semana. Essa análise ajuda a justificar baselines sazonais simples, como média por weekday e seasonal naive.

In [ ]:
weekday_names = {
    0: "segunda",
    1: "terça",
    2: "quarta",
    3: "quinta",
    4: "sexta",
    5: "sábado",
    6: "domingo",
}

weekday_sales = daily_sales.assign(
    weekday=daily_sales["date"].dt.weekday,
    dia_semana=daily_sales["date"].dt.weekday.map(weekday_names),
).groupby(["weekday", "dia_semana"], as_index=False).agg(media_vendas=("sales", "mean"))

fig = px.bar(
    weekday_sales,
    x="dia_semana",
    y="media_vendas",
    title="Média de vendas por dia da semana",
    labels={"dia_semana": "dia da semana", "media_vendas": "média de vendas"},
)
fig.show()
weekday_sales

## 7. Mix por família e por loja

O dashboard precisa apoiar priorização. Por isso, entender o mix ajuda a identificar quais famílias e lojas mais influenciam o resultado consolidado.

In [ ]:
family_mix = (
    train.groupby("family", observed=True, as_index=False)
    .agg(sales=("sales", "sum"), promo_items=("onpromotion", "sum"))
    .sort_values("sales", ascending=False)
)
family_mix["share"] = family_mix["sales"] / family_mix["sales"].sum()

family_mix.head(12)

In [ ]:
fig = px.bar(
    family_mix.head(12).sort_values("sales"),
    x="sales",
    y="family",
    orientation="h",
    title="Top 12 famílias por volume histórico",
    labels={"sales": "vendas", "family": "família"},
)
fig.show()

In [ ]:
stores_enriched = stores.copy()
stores_enriched["store"] = (
    stores_enriched["city"] + " - Loja " + stores_enriched["store_nbr"].astype(str)
)

store_mix = (
    train.merge(
        stores_enriched[["store_nbr", "store", "city", "state", "type", "cluster"]],
        on="store_nbr",
        how="left",
    )
    .groupby(["store_nbr", "store", "city", "state"], as_index=False)
    .agg(sales=("sales", "sum"), promo_items=("onpromotion", "sum"))
    .sort_values("sales", ascending=False)
)

store_mix.head(12)

## 8. Promoções

A coluna `onpromotion` indica a quantidade de itens em promoção naquela data, loja e família. Ela não é preço, desconto ou margem, mas é um sinal operacional importante.

No dashboard, esse sinal aparece como participação promocional e também influencia risco operacional.

In [ ]:
promo_summary = train.assign(em_promocao=train["onpromotion"] > 0).groupby("em_promocao").agg(
    linhas=("sales", "size"),
    vendas_totais=("sales", "sum"),
    venda_media=("sales", "mean"),
    itens_promocao_medios=("onpromotion", "mean"),
)
promo_summary["share_vendas"] = (
    promo_summary["vendas_totais"] / promo_summary["vendas_totais"].sum()
)
promo_summary

In [ ]:
promo_daily = daily_sales.copy()
correlation = promo_daily[["promoted_items", "sales"]].corr().iloc[0, 1]
print(f"Correlação diária entre itens em promoção e vendas: {correlation:.3f}")

fig = px.scatter(
    promo_daily,
    x="promoted_items",
    y="sales",
    title="Relação diária entre itens em promoção e vendas",
    labels={"promoted_items": "itens em promoção", "sales": "vendas"},
)
fig.show()

## 9. Conclusões da EDA

Principais aprendizados para a modelagem:

1. A série é naturalmente hierárquica: total da empresa, loja, família e combinação loja-família.
2. A granularidade operacional do forecast deve ser `data + loja + família`.
3. Há sinal de sazonalidade por dia da semana, justificando baselines sazonais simples.
4. Promoção é um sinal relevante, mas precisa ser interpretado com cuidado: `onpromotion` é quantidade de itens em promoção, não valor de desconto.
5. Como não existe preço ou margem, o projeto trata `sales` como volume de vendas planejadas.
6. Para decisão executiva, o forecast precisa ser acompanhado de risco, confiança e recomendação operacional, não apenas uma linha prevista.

Próximo passo: construir baselines e um candidato interpretável de forecast no notebook `02_modelagem_baselines.ipynb`.